In [29]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel,Field
import operator

In [30]:
load_dotenv()

True

In [31]:
model = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash' )

In [32]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description='Detailed feedback for essay')
    score: int = Field(description='Score out of 10',ge=0,le=10)

In [33]:
structured_model = model.with_structured_output(EvaluationSchema)

In [34]:
essay = """# The Importance of Artificial Intelligence in Everyday Life

## Introduction

Artificial Intelligence (AI) is one of the most significant technological developments of the modern world. It refers to the ability of machines and computer systems to perform tasks that typically require human intelligence, such as learning, problem-solving, decision-making, and understanding language. Today, AI has become an important part of our daily lives, influencing how we communicate, work, study, and interact with technology.

## Applications of Artificial Intelligence

Artificial Intelligence is used in various fields, including education, healthcare, transportation, banking, and business. In education, AI-powered tools help students understand difficult concepts, improve their writing, and learn at their own pace. In healthcare, AI assists doctors in analyzing medical images, identifying potential diseases, and improving patient care.

Similarly, virtual assistants, recommendation systems, and navigation applications use AI to make everyday activities more convenient. Businesses also use AI to analyze customer preferences, automate repetitive tasks, and make better decisions. These applications demonstrate how AI can improve efficiency and productivity.

## Advantages of Artificial Intelligence

One of the major advantages of AI is its ability to process large amounts of information quickly. It can automate repetitive tasks, reduce human effort, and help organizations save time and resources. AI can also operate continuously without becoming tired, making it useful in situations that require constant monitoring.

Furthermore, AI can support innovation by helping researchers discover new solutions to complex problems. When used responsibly, it can improve the quality of education, healthcare, and public services.

## Challenges and Limitations

Despite its benefits, Artificial Intelligence also presents several challenges. Automation may reduce the demand for certain jobs, requiring workers to develop new skills. AI systems can also produce inaccurate information, reflect biases in their training data, and create concerns about privacy and data security.

Another important concern is the excessive dependence on technology. People must continue developing their critical thinking, creativity, and decision-making abilities rather than relying entirely on machines. Therefore, appropriate regulations, human supervision, and responsible development are essential.

## Conclusion

In conclusion, Artificial Intelligence is transforming the modern world by making many activities faster, easier, and more efficient. Although it presents challenges, its potential benefits are significant when the technology is used responsibly. The future of AI will depend not only on technological progress but also on how wisely humans choose to use it. By balancing innovation with ethical responsibility, society can use Artificial Intelligence to create a better future.
"""

In [35]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
structured_model.invoke(prompt)

EvaluationSchema(feedback='The essay demonstrates excellent language quality. The vocabulary is precise and appropriate for the topic, effectively conveying complex ideas without being overly academic or simplistic. Sentence structures are varied, contributing to a smooth and engaging reading experience. There are no noticeable grammatical errors or spelling mistakes, and the essay maintains a clear, concise, and objective tone throughout. The transitions between paragraphs are seamless, ensuring a logical flow of ideas from introduction to conclusion. The clarity of expression makes the essay very easy to understand and follow, making it a highly effective piece of writing.', score=9)

In [36]:
class UPSCState(TypedDict):
    essay :str
    language_feedback : str
    analysis_feedback : str
    clarity_feedback : str
    overall_feedback: str
    individual_score : Annotated[list[int],operator.add]
    avg_score : float

In [37]:
def evaluate_language(state:UPSCState):
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'language_feedback':output.feedback, 'individual_score':[output.score]}

In [43]:
def evaluate_analysis(state:UPSCState):
    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {'analysis_feedback':output.feedback, 'individual_score':[output.score]}

In [39]:
def evaluate_thought(state:UPSCState):
    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state["essay"]}'
    output = structured_model.invoke(prompt)

    return {' clarity_feedback':output.feedback, 'individual_score':[output.score]}

In [40]:
def final_evaluation(state:UPSCState):
   prompt = f'Based on the following feedbacks create a summarized feedback \n language feedback- {state["language_feedback"]} \n analysis feedback- {state["analysis_feedback"]} \n clarity feedback-{state["clarity_feedback"]}'
   overall_feedback = model.invoke(prompt).content

   avg_score = sum(state['individual_score'])/len(state['individual_score'])

   return {'overall_feedback': overall_feedback , 'avg_score': avg_score}

In [44]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evlauate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)

graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evlauate_analysis')
graph.add_edge(START,'evaluate_thought')

graph.add_edge('evaluate_language','final_evaluation')
graph.add_edge('evlauate_analysis','final_evaluation')
graph.add_edge('evaluate_thought','final_evaluation')

graph.add_edge('final_evaluation',END)

workflow = graph.compile()

In [46]:
initial_state = {
    'essay' : essay
}

workflow.invoke(initial_state)

GoogleRateLimitError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 18h27m53.064097016s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 